# Episode 9: Generalization: Overfitting, Validation, Regularization
<a href="https://colab.research.google.com/github/Kinvecta/how-llms-work/blob/main/notebooks/Episode%2009%20-%20Generalization.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

**Companion notebook** for the series *How Large Language Models Work: From Math to Transformers*.

This notebook reproduces the Math Lab numbers from Episode 9, one step per section. It uses only `numpy` (already installed in Colab), so choose **Runtime, Run all** and nothing else is needed.

Covers: the running-sentence memorizer versus learner, the exact bias-variance toy, L2 and weight decay, and the dropout check. Toy numbers are illustrative.

Every value that appears in the episode is checked with an `assert`. If a cell fails, a number in the script and the math have drifted apart.

In [1]:
import numpy as np
from itertools import product
print("numpy", np.__version__)

numpy 2.5.3


## Section 3: the memorizer versus the learner
The memorizer is Episode 8's slider on one training term (N = 1): η = 5, ten steps, other scores fixed (R = 0.70). Validation is N = 20 held-out terms in exactly the language's mix.

In [2]:
R = 0.70
p_lang = np.array([0.30, 0.20, 0.15, 0.35])        # mat, floor, sofa, other (Episodes 1 and 4)
words = ["mat", "floor", "sofa", "other"]

def p_mat(z):  return np.exp(z) / (np.exp(z) + R)
z = np.log(0.30)
def step(z, eta):  return z + eta * (1 - p_mat(z))       # z <- z - eta * dL/dz, dL/dz = -(1 - p)

for _ in range(10):
    z = step(z, 5)
pm = p_mat(z)
rest = (1 - pm) * p_lang[1:] / p_lang[1:].sum()          # the other three share what is left, in their old ratio
p_mem = np.concatenate([[pm], rest])
print("memorizer:", dict(zip(words, np.round(p_mem, 4))))
assert round(pm, 3) == 0.984 and np.round(rest, 4).tolist() == [0.0046, 0.0034, 0.0080]
assert round(-np.log(pm), 3) == 0.016                    # training loss, N = 1

counts = np.array([6, 4, 3, 7])                           # N = 20 held-out terms
assert counts.sum() == 20 and np.allclose(counts / 20, p_lang)
def L_val(p):  return -(counts * np.log(p)).sum() / counts.sum()
def pp(L):     return np.exp(L)

H = -(p_lang * np.log(p_lang)).sum()
L_learner, L_mem = L_val(p_lang), L_val(p_mem)
print(f"learner:   L_train = 1.204, L_val = {L_learner:.3f}, PP = {pp(L_learner):.2f}")
print(f"memorizer: L_train = {-np.log(pm):.3f}, L_val = {L_mem:.3f}, PP = {pp(L_mem):.1f}")
assert round(H, 3) == 1.335 and round(L_learner, 3) == 1.335 and round(pp(L_learner), 2) == 3.80
assert round(L_mem, 3) == 3.624 and round(pp(L_mem), 1) == 37.5
kl = (p_lang * np.log(p_lang / p_mem)).sum()
print(f"H + KL = {H:.3f} + {kl:.4f} = {H + kl:.3f}")
assert round(kl, 4) == 2.2885 and abs(H + kl - L_mem) < 1e-12

# L_val after one step of the slider
pm1 = p_mat(np.log(0.30) + 5 * 0.70)           # one step: z rises by eta * 0.70
p1 = np.concatenate([[pm1], (1 - pm1) * p_lang[1:] / p_lang[1:].sum()])
print(f"L_val after one step: {L_val(p1):.3f}")
assert round(L_val(p1), 3) == 2.649

memorizer: {'mat': np.float64(0.984), 'floor': np.float64(0.0046), 'sofa': np.float64(0.0034), 'other': np.float64(0.008)}
learner:   L_train = 1.204, L_val = 1.335, PP = 3.80
memorizer: L_train = 0.016, L_val = 3.624, PP = 37.5
H + KL = 1.335 + 2.2885 = 3.624
L_val after one step: 2.649


## Math Lab, Steps 1 and 2: the bias-variance decomposition, exact toy
Inputs u = 1, 2, 3; truth g(u) = 2u; noise ε = ±1 with probability ½ (τ² = 1): 8 equally likely training sets. Held-out input u = 4, fresh noise: 16 cases.

In [3]:
u = np.array([1.0, 2.0, 3.0]); g = lambda t: 2 * t; tau2 = 1.0
u_test = 4.0

def fit_const(y):  return np.full(3, y.mean())
def fit_poly(deg):
    def f(y):
        c = np.polyfit(u, y, deg)
        return lambda t: np.polyval(c, t)
    return f
models = {
    "Constant": lambda y: (lambda t: y.mean()),
    "Line":     fit_poly(1),
    "Parabola": fit_poly(2),
}
sets = [np.array([g(t) + e for t, e in zip(u, eps)]) for eps in product([-1, 1], repeat=3)]
assert len(sets) == 8

want = {  # mean prediction, bias^2, variance, Err at u = 4, expected training MSE
    "Constant": (4, 16, 1/3, 17.33, 3.33),
    "Line":     (8, 0, 7/3, 3.33, 0.33),
    "Parabola": (8, 0, 19, 20, 0.0),
}
print("model     | mean pred | bias^2 | variance | tau^2 | Err(4)  | E[train MSE]")
for name, make in models.items():
    preds = np.array([make(y)(u_test) for y in sets])
    mean_pred = preds.mean()
    bias2 = (g(u_test) - mean_pred) ** 2
    var = preds.var()
    err_formula = tau2 + bias2 + var
    # enumerate all 16 cases: 8 training sets x 2 test noises
    err_enum = np.mean([(g(u_test) + e - make(y)(u_test)) ** 2 for y in sets for e in (-1, 1)])
    train_mse = np.mean([np.mean((y - np.array([make(y)(t) for t in u])) ** 2) for y in sets])
    print(f"{name:<9} | {mean_pred:9.2f} | {bias2:6.2f} | {var:8.3f} | {tau2:5.1f} | {err_formula:7.2f} | {train_mse:.2f}")
    m, b2, v, e, t = want[name]
    assert abs(mean_pred - m) < 1e-9 and abs(bias2 - b2) < 1e-9 and abs(var - v) < 1e-9
    assert round(err_formula, 2) == e and abs(err_formula - err_enum) < 1e-9
    assert round(train_mse, 2) == t

# each prediction at 4 is a fixed weighted sum of the labels; variance = tau^2 * sum of squared weights
W = {"Constant": [1/3, 1/3, 1/3], "Line": [-2/3, 1/3, 4/3], "Parabola": [1, -3, 3]}
for name, w in W.items():
    w = np.array(w)
    for y in sets:
        assert abs(models[name](y)(u_test) - w @ y) < 1e-9
    assert abs(np.sum(w ** 2) * tau2 - {"Constant": 1/3, "Line": 7/3, "Parabola": 19}[name]) < 1e-9
print("weights and variances check out.")

model     | mean pred | bias^2 | variance | tau^2 | Err(4)  | E[train MSE]
Constant  |      4.00 |  16.00 |    0.333 |   1.0 |   17.33 | 3.33
Line      |      8.00 |   0.00 |    2.333 |   1.0 |    3.33 | 0.33
Parabola  |      8.00 |   0.00 |   19.000 |   1.0 |   20.00 | 0.00
weights and variances check out.


In [4]:
# Check question (a): copy the third label, weights (0, 0, 1)
w = np.array([0.0, 0.0, 1.0])
mean_pred = w @ g(u)
bias2 = (g(u_test) - mean_pred) ** 2
var = tau2 * np.sum(w ** 2)
print(f"mean prediction {mean_pred}, bias^2 {bias2}, variance {var}, Err {tau2 + bias2 + var}")
assert mean_pred == 6 and bias2 == 4 and var == 1 and tau2 + bias2 + var == 6

mean prediction 6.0, bias^2 4.0, variance 1.0, Err 6.0


## Math Lab, Step 3: L2 and weight decay

In [5]:
# L(z) = (z - 2)^2 + lam z^2: the minimizer moves to 2 / (1 + lam)
for lam, want in ((0, 2.0), (0.25, 1.6), (1, 1.0), (9, 0.2)):
    zs = np.linspace(-1, 4, 500001)
    argmin = zs[np.argmin((zs - 2) ** 2 + lam * zs ** 2)]
    print(f"lambda = {lam}: minimizer {argmin:.3f}")
    assert abs(argmin - want) < 1e-4 and abs(2 / (1 + lam) - want) < 1e-12

# stable range: update z <- (1 - 2 eta (1 + lam)) z + 4 eta, so eta < 1/(1 + lam)
lam, eta = 1, 0.6
assert round(1 - 2 * eta * (1 + lam), 1) == -1.4 and round(1 - 2 * eta, 1) == -0.2

# weight decay: theta <- theta - eta (grad L + 2 lam theta) = (1 - 2 eta lam) theta - eta grad L
rng = np.random.default_rng(0)
theta, gL = rng.normal(size=5), rng.normal(size=5)
eta, lam = 0.1, 0.3
a = theta - eta * (gL + 2 * lam * theta)
b = (1 - 2 * eta * lam) * theta - eta * gL
assert np.allclose(a, b)

# check question (b): L(z) = (z - 2)^2 + z^2: gradient 4z - 4, minimizer 1, update (1 - 4 eta) z + 4 eta
z, eta = 5.0, 0.25
assert abs((1 - 4 * eta) * z + 4 * eta - 1) < 1e-12          # lands on 1 in one step at eta = 1/4
for eta, converges in ((0.45, True), (0.55, False)):
    z = 5.0
    for _ in range(400):
        z = (1 - 4 * eta) * z + 4 * eta
    assert (abs(z - 1) < 1e-6) == converges
print("minimizer 1; stable for 0 < eta < 1/2; lands on 1 in one step at eta = 1/4.")

lambda = 0: minimizer 2.000
lambda = 0.25: minimizer 1.600
lambda = 1: minimizer 1.000
lambda = 9: minimizer 0.200
minimizer 1; stable for 0 < eta < 1/2; lands on 1 in one step at eta = 1/4.


## Math Lab, Step 4: the dropout check
A unit adds 3·h₁ − 1·h₂, with h = (2, 4) and ζ = ½ (keep probability).

In [6]:
h = np.array([2.0, 4.0]); w = np.array([3.0, -1.0]); zeta = 0.5
outs = [float(w @ (np.array(m) * h)) for m in product([1, 0], repeat=2)]   # masks (1,1), (1,0), (0,1), (0,0)
print("four masks:", outs, " mean", np.mean(outs))
assert outs == [2.0, 6.0, -4.0, 0.0] and np.mean(outs) == 1.0

test_out = float((zeta * w) @ h)                  # full network, outgoing weights times zeta
print("test, weights halved:", test_out)
assert test_out == 1.0

inv = [float(w @ (np.array(m) * h / zeta)) for m in product([1, 0], repeat=2)]   # kept values / zeta
print("inverted dropout:", inv, " mean", np.mean(inv), " unscaled test output", float(w @ h))
assert inv == [4.0, 12.0, -8.0, 0.0] and np.mean(inv) == 2.0 and float(w @ h) == 2.0

relu = lambda x: max(x, 0.0)
mask_avg = np.mean([relu(o) for o in outs])
print("ReLU: mask average", mask_avg, " ReLU of the scaled test output", relu(test_out))
assert mask_avg == 2.0 and relu(test_out) == 1.0

four masks: [2.0, 6.0, -4.0, 0.0]  mean 1.0
test, weights halved: 1.0
inverted dropout: [4.0, 12.0, -8.0, 0.0]  mean 2.0  unscaled test output 2.0
ReLU: mask average 2.0  ReLU of the scaled test output 1.0


In [7]:
print("All Episode 9 checks passed.")

All Episode 9 checks passed.
